In [ ]:
import time
import joblib
import pandas as pd
import numpy as np
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score

# -----------------------------
# Load data (same as training)
# -----------------------------
data = pd.read_csv("../../../data/raw/kathmandu_full_raw_2023_2024.csv")
data["time"] = pd.to_datetime(data["time"])
data = data.sort_values("time").reset_index(drop=True)
data = data.set_index("time")
data.index.freq = "h"

split = int(np.ceil(0.8 * len(data)))
train = data.iloc[:split]
test = data.iloc[split:]

# -----------------------------
# Load saved model
# -----------------------------
model = joblib.load("arima_model.pkl")

# -----------------------------
# Rolling 12-hour-ahead evaluation
# -----------------------------
HORIZON = 12
test_pm25 = test["pm2_5"].reset_index(drop=True)

preds = []
for i in range(len(test_pm25) - HORIZON + 1):
    forecast = model.predict(n_periods=HORIZON)
    preds.append(forecast[-1])            # <-- fixed: plain indexing, not .iloc
    model.update(test_pm25.iloc[[i]])

preds = np.array(preds)
true = test_pm25.iloc[HORIZON - 1 : HORIZON - 1 + len(preds)].values

rmse = root_mean_squared_error(true, preds)
mae = mean_absolute_error(true, preds)
r2 = r2_score(true, preds)

print(f"RMSE (12h horizon) : {rmse:.4f}")
print(f"MAE  (12h horizon) : {mae:.4f}")
print(f"R2   (12h horizon) : {r2:.4f}")